<a href="https://colab.research.google.com/github/suryanshdev2023/tableau_text_summarizer_extension/blob/main/WB_persona_LLM_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/01_consulting/05_WB/01_persona_LLM_tuning

/content/drive/.shortcut-targets-by-id/1ExKZ-pSZ8aOGWi5jDk41KgvOPfBU0pN_/01_consulting/05_WB/01_persona_LLM_tuning


In [4]:
!pip install -q -U -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.7/530.7 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.1/366.1 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.9/169.9 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.5/196.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.

In [4]:
# 1. Nuke broken torch + bnb + caches
#!pip uninstall -y -q torch torchvision torchaudio bitsandbytes
#!pip cache purge

# 2. Install torch built for CUDA 12.8 (FIRST, before anything else)
!pip install -q --index-url https://download.pytorch.org/whl/cu128 \
    torch torchvision torchaudio

# 3. Install the rest of requirements (no torch in there anymore)
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 12.8 MB/s eta 0:00:00


In [5]:
import torch, bitsandbytes as bnb
from bitsandbytes.nn import Linear4bit
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("bnb :", bnb.__version__)
print("Linear4bit loads OK")

ERROR:bitsandbytes.cextension:Could not load bitsandbytes native library: /lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4.32' not found (required by /usr/local/lib/python3.12/dist-packages/bitsandbytes/libbitsandbytes_cpu.so)
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 85, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 72, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: /lib/x86_64-linux-gnu/libstdc++.so.6: version `GLIBCXX_3.4

AssertionError: Torch not compiled with CUDA enabled

In [4]:
!python test_persona.py \
    --base_model DiscoResearch/DiscoLM_German_7b_v1 \
    --segment "Heavy User 25-39J" \
    --question "Was verbindet ihr mit DMAX?"


=== Segment: Heavy User 25-39J ===
System prompt: Du bist ein männlicher DMAX-Heavy-User zwischen 25 und 39 Jahren. Du schaust DMAX regelmäßig und intensiv, interessierst dich für Autos, Technik, Handwerk, Abenteuer und authentische Reality-Formate. Antworte in lockerem, norddeutsch-umgangssprachlichem Deutsch, kurz, direkt und ehrlich, so wie in einem Gruppeninterview.

--- BASE ---
config.json: 100% 654/654 [00:00<00:00, 5.23MB/s]
2026-04-22 21:03:27.116209: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776891807.342634   11438 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776891807.406854   11438 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered

In [6]:
!python test_persona.py \
    --base_model DiscoResearch/DiscoLM_German_7b_v1 \
    --adapter adapters/disco-persona-v1 \
    --segment "Heavy User 25-39J" \
    --compare

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]

=== Segment: Heavy User 25-39J ===
System prompt: Du bist ein männlicher DMAX-Heavy-User zwischen 25 und 39 Jahren. Du schaust DMAX regelmäßig und intensiv, interessierst dich für Autos, Technik, Handwerk, Abenteuer und authentische Reality-Formate. Antworte in lockerem, norddeutsch-umgangssprachlichem Deutsch, kurz, direkt und ehrlich, so wie in einem Gruppeninterview.

--- BASE (no adapter) ---
2026-04-23 05:57:51.623003: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-23 05:57:52.092754: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-04-23 05:57:52.321025: E external/local_xla/xla

In [14]:
!python train_qlora.py \
    --base_model DiscoResearch/DiscoLM_German_7b_v1 \
    --output_dir adapters/disco-persona-v1 \
    --epochs 3 --batch_size 2 --grad_accum 8 --max_seq_len 1024 --lr 2e-4

2026-04-22 21:54:39.172601: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776894879.197250   25125 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776894879.209165   25125 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776894879.235473   25125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776894879.235515   25125 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776894879.235523   25125 computation_placer.cc:177] computation placer alr

In [13]:
!rm -rf adapters/disco-persona-v1